# Paper 2 · V2: censored-training test (Derm7pt → EASY)

**Before running, in the right-hand panel:**
1. **Accelerator:** GPU T4 x2
2. **Internet:** ON (downloads ImageNet weights and the 248 EASY images)
3. **Input:** the same `derm7pt` dataset you used for the EASY replication job (Add Input)

Then **Save Version → Save & Run All (Commit)**. Expected time: about 1.5 hours (9 trainings of about 18 minutes each, split over 2 GPUs).

Cell 1 writes the training script. Cell 2 checks cell 1, downloads EASY, trains 9 EfficientNet-B4 models on all 1011 Derm7pt images (3 seeds × censoring 0, 0.5, 1), and predicts on EASY.

**Download:** `p2_v2.zip` (27 small files) from the Output tab. The `weights/` folder can stay in Kaggle.


In [ ]:
%%writefile /kaggle/working/train_v2_censored.py
# Paper 2 - V2: censored-training test (pre-registered in the D1 protocol,
# "Pre-registration of V2"). Same recipe as the EASY replication job, except
# that the Derm7pt CONCEPT labels used for training are censored with
# probability PI: on naevi, melanoma-side categories are set to "absent";
# on melanomas, naevus-side categories are set to "absent". Diagnosis labels
# and all other lesions are untouched.
# 1) Train the concept encoder on ALL 1011 Derm7pt images (censored labels).
# 2) Save the weights and the censored label table.
# 3) Predict concept probabilities + black-box melanoma probability on EASY.
# Settings come from environment variables set by cell 2:
#   P2_JOBS      e.g. "effb4:0:0,effb4:0:1"  (backbone:seed:pi)
#   P2_EASY_DIR  folder with the EASY ISIC_*.jpg images
#   P2_INPUT     Kaggle input root (default /kaggle/input)
#   P2_OUT       output folder (default /kaggle/working)
#   P2_SMOKE=1   tiny local test only
import os
import glob
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
import torchvision.models as tvm
from sklearn.metrics import roc_auc_score

SMOKE = os.environ.get("P2_SMOKE", "0") == "1"
INPUT_ROOT = os.environ.get("P2_INPUT", "/kaggle/input")
OUT_DIR = os.environ.get("P2_OUT", "/kaggle/working")
EASY_DIR = os.environ.get("P2_EASY_DIR", "/tmp/easy_images")
JOB_TEXT = os.environ.get("P2_JOBS", "effb4:0:0")
JOBS = []
for j in JOB_TEXT.split(","):
    if j:
        name, sd, pi = j.split(":")
        JOBS.append((name, int(sd), float(pi)))
EPOCHS = 2 if SMOKE else 40
# identical to the confirmation runs (kaggle_dump_concepts.py)
RECIPE = {
    "effb4": dict(IMG=380, BS=16, LR=3e-4, WD=1e-4),
    "convnext_tiny": dict(IMG=384, BS=16, LR=1e-4, WD=5e-2),
}
CONCEPTS = {
    "pigment_network": ["absent", "typical", "atypical"],
    "blue_whitish_veil": ["absent", "present"],
    "vascular_structures": [
        "absent", "arborizing", "comma", "hairpin",
        "within regression", "wreath", "dotted", "linear irregular"],
    "streaks": ["absent", "regular", "irregular"],
    "pigmentation": [
        "absent", "diffuse regular", "localized regular",
        "diffuse irregular", "localized irregular"],
    "dots_and_globules": ["absent", "regular", "irregular"],
    "regression_structures": [
        "absent", "blue areas", "white areas", "combinations"],
}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"
if not SMOKE and DEVICE != "cuda":
    raise SystemExit("No GPU visible to this job.")
os.makedirs(os.path.join(OUT_DIR, "weights"), exist_ok=True)


def key_of(path):
    parts = path.replace("\\", "/").split("/")
    return "/".join(parts[-2:]).lower()


# ---------------- Derm7pt (training data) ----------------
pattern = os.path.join(INPUT_ROOT, "**", "meta.csv")
meta = None
for mp in sorted(glob.glob(pattern, recursive=True)):
    d = pd.read_csv(mp)
    if len(d) == 1011 and "derm" in d.columns:
        meta = d
        print("Derm7pt meta.csv:", mp, flush=True)
        break
if meta is None:
    raise SystemExit("Derm7pt meta.csv (1011 rows) not found under "
                     + INPUT_ROOT + " -> add the derm7pt dataset")
index = {}
all_files = glob.glob(os.path.join(INPUT_ROOT, "**", "*"), recursive=True)
for p in all_files:
    if p.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
        index[key_of(p)] = p
paths = [index.get(key_of(d)) for d in meta["derm"]]
missing = [d for d, p in zip(meta["derm"], paths) if p is None]
if missing:
    raise SystemExit(str(len(missing)) + " Derm7pt images missing")
meta["img_path"] = paths
meta["mel"] = meta["diagnosis"].str.contains("melanoma").astype(int)
for c, cats in CONCEPTS.items():
    bad = set(meta[c]) - set(cats)
    assert not bad, "unexpected categories in " + c
    lookup = {k: i for i, k in enumerate(cats)}
    meta[c + "__y"] = meta[c].map(lookup)
n_mel = int(meta.mel.sum())
print("Derm7pt:", len(meta), "images,", n_mel, "melanomas", flush=True)

MEL_SIDE = {"pigment_network": ["atypical"], "streaks": ["irregular"],
            "pigmentation": ["diffuse irregular", "localized irregular"],
            "dots_and_globules": ["irregular"],
            "regression_structures": ["blue areas", "white areas",
                                      "combinations"],
            "blue_whitish_veil": ["present"],
            "vascular_structures": ["dotted", "linear irregular"]}
NEV_SIDE = {"pigment_network": ["typical"], "streaks": ["regular"],
            "pigmentation": ["diffuse regular", "localized regular"],
            "dots_and_globules": ["regular"]}


def censor(meta, pi):
    """Censored copy of the Derm7pt labels. One uniform draw per
    (image, criterion), seed fixed by pi, identical for every seed."""
    m = meta.copy()
    rng = np.random.default_rng(2026 + int(round(pi * 100)))
    dx = m["diagnosis"].str.lower()
    is_mel = dx.str.contains("melanoma").values
    is_nev = (dx.str.contains("nevus") & ~dx.str.contains("melanoma")).values
    changed = 0
    for c, cats in CONCEPTS.items():
        u = rng.random(len(m))
        drop = np.zeros(len(m), bool)
        if c in MEL_SIDE:
            drop |= is_nev & m[c].isin(MEL_SIDE[c]).values
        if c in NEV_SIDE:
            drop |= is_mel & m[c].isin(NEV_SIDE[c]).values
        drop &= u < pi
        changed += int(drop.sum())
        m.loc[drop, c] = "absent"
        lookup = {k: i for i, k in enumerate(cats)}
        m[c + "__y"] = m[c].map(lookup)
    return m, changed

# ---------------- EASY (external test images) ----------------
pattern = os.path.join(EASY_DIR, "**", "ISIC_*.jp*g")
easy_paths = sorted(glob.glob(pattern, recursive=True))
easy_ids = [os.path.splitext(os.path.basename(p))[0] for p in easy_paths]
if not SMOKE and len(set(easy_ids)) != 248:
    raise SystemExit("expected 248 EASY images, found "
                     + str(len(set(easy_ids))))
easy = pd.DataFrame({"isic_id": easy_ids, "img_path": easy_paths})
easy = easy.drop_duplicates("isic_id")
# Quick gate check only (Benign/Malignant from ISIC). The official
# gate is computed later on CPU from the repository metadata.
easy_labels = None
LABEL_COLS = ["diagnosis_1", "benign_malignant"]
pattern = os.path.join(EASY_DIR, "**", "*.csv")
for mp in glob.glob(pattern, recursive=True):
    md = pd.read_csv(mp)
    md.columns = [c.lower() for c in md.columns]
    cols = [c for c in LABEL_COLS if c in md.columns]
    if "isic_id" in md.columns and cols:
        md = md.drop_duplicates("isic_id").set_index("isic_id")
        lab = md[cols[0]].astype(str).str.lower()
        easy_labels = lab.map({"malignant": 1, "benign": 0})
        break
label_note = "found" if easy_labels is not None else "not found"
print("EASY:", len(easy), "images; labels for gate:", label_note,
      flush=True)

MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]


def make_tf(img, train):
    steps = [T.Resize((img, img))]
    if train:
        steps += [T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
                  T.RandomRotation(20),
                  T.ColorJitter(0.1, 0.1, 0.1, 0.02)]
    steps += [T.ToTensor(), T.Normalize(MEAN, STD)]
    return T.Compose(steps)


class TrainDS(Dataset):
    def __init__(self, df, tf):
        self.df = df.reset_index(drop=True)
        self.tf = tf

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        y = torch.tensor([r[c + "__y"] for c in CONCEPTS],
                         dtype=torch.long)
        x = self.tf(Image.open(r.img_path).convert("RGB"))
        return x, y, torch.tensor(float(r.mel))


class PredDS(Dataset):
    def __init__(self, df, tf):
        self.df = df.reset_index(drop=True)
        self.tf = tf

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        return self.tf(Image.open(r.img_path).convert("RGB")), i


class Net(nn.Module):
    def __init__(self, bb, nf):
        super().__init__()
        self.bb = bb
        self.drop = nn.Dropout(0.3)
        self.heads = nn.ModuleList(
            [nn.Linear(nf, len(v)) for v in CONCEPTS.values()])
        self.mel = nn.Linear(nf, 1)

    def forward(self, x):
        h = self.drop(self.bb(x))
        return [hd(h) for hd in self.heads], self.mel(h).squeeze(1)


def build(backbone):
    try:
        if backbone == "convnext_tiny":
            w = None if SMOKE else tvm.ConvNeXt_Tiny_Weights.IMAGENET1K_V1
            bb = tvm.convnext_tiny(weights=w)
            nf = bb.classifier[2].in_features
            bb.classifier[2] = nn.Identity()
        elif SMOKE:
            bb = tvm.efficientnet_b0(weights=None)
            nf = bb.classifier[1].in_features
            bb.classifier = nn.Identity()
        else:
            w = tvm.EfficientNet_B4_Weights.IMAGENET1K_V1
            bb = tvm.efficientnet_b4(weights=w)
            nf = bb.classifier[1].in_features
            bb.classifier = nn.Identity()
    except Exception as e:
        raise SystemExit("Could not get ImageNet weights: " + str(e)
                         + " -> turn Internet ON")
    return Net(bb, nf).to(DEVICE)


@torch.no_grad()
def predict(model, df, img, bs):
    model.eval()
    rows = []
    loader = DataLoader(PredDS(df, make_tf(img, False)), bs,
                        shuffle=False, num_workers=2)
    for x, idx in loader:
        with torch.autocast(DEVICE, enabled=USE_AMP):
            outs, m = model(x.to(DEVICE))
        probs = [F.softmax(o.float(), 1).cpu().numpy() for o in outs]
        pm = torch.sigmoid(m.float()).cpu().numpy()
        for b in range(len(idx)):
            r = {"isic_id": df.iloc[int(idx[b])].isic_id,
                 "p_mel_bb": float(pm[b])}
            for (c, cats), P in zip(CONCEPTS.items(), probs):
                for k, cat in enumerate(cats):
                    r[c + "__" + cat] = float(P[b, k])
            rows.append(r)
    return pd.DataFrame(rows)


for backbone, seed, PI in JOBS:
    cfg = RECIPE[backbone]
    IMG, BS = (64, 8) if SMOKE else (cfg["IMG"], cfg["BS"])
    tag = "v2_pi%03d_%s_seed%d" % (int(round(PI * 100)), backbone, seed)
    pred_csv = os.path.join(OUT_DIR, "easy_preds_" + tag + ".csv")
    tmeta, n_changed = censor(meta, PI)
    print("==", tag, ": censoring pi =", PI, "changed", n_changed,
          "concept labels", flush=True)
    tmeta.drop(columns=["img_path"]).to_csv(
        os.path.join(OUT_DIR, "train_labels_" + tag + ".csv"), index=False)
    if os.path.exists(pred_csv):
        print("skip (exists):", pred_csv, flush=True)
        continue
    # full-data runs have no fold, so the seed is used directly
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    model = build(backbone)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["LR"],
                            weight_decay=cfg["WD"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)
    scaler = torch.amp.GradScaler(enabled=USE_AMP)
    dl = DataLoader(TrainDS(tmeta, make_tf(IMG, True)), BS,
                    shuffle=True, num_workers=2, drop_last=True)
    print("==", tag, ": training on all", len(tmeta),
          "Derm7pt images,", EPOCHS, "epochs, img", IMG, flush=True)
    log = []
    for ep in range(1, EPOCHS + 1):
        model.train()
        t0 = time.time()
        tot = 0.0
        for x, y, m in dl:
            x, y, m = x.to(DEVICE), y.to(DEVICE), m.to(DEVICE)
            with torch.autocast(DEVICE, enabled=USE_AMP):
                outs, pm = model(x)
                ce = [F.cross_entropy(o, y[:, j])
                      for j, o in enumerate(outs)]
                bce = F.binary_cross_entropy_with_logits(pm, m)
                loss = sum(ce) + bce
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            tot += loss.item()
        sched.step()
        msg = {"epoch": ep, "loss": round(tot / max(len(dl), 1), 4),
               "sec": round(time.time() - t0, 1)}
        log.append(msg)
        if ep in (1, EPOCHS) or ep % 10 == 0:
            print("[" + tag + "]", msg, flush=True)
    state = {}
    for k, v in model.state_dict().items():
        state[k] = v.half() if v.is_floating_point() else v
    torch.save(state, os.path.join(OUT_DIR, "weights", tag + ".pt"))
    P = predict(model, easy, IMG, BS * 2)
    P.insert(0, "seed", seed)
    P.insert(0, "backbone", backbone)
    P.to_csv(pred_csv, index=False)
    log_csv = os.path.join(OUT_DIR, "log_full_" + tag + ".csv")
    pd.DataFrame(log).to_csv(log_csv, index=False)
    if easy_labels is not None:
        y = P.isic_id.map(easy_labels)
        ok = y.notna()
        if ok.sum() > 10 and y[ok].nunique() == 2:
            auc = roc_auc_score(y[ok], P.p_mel_bb[ok])
            print("[" + tag + "] EASY gate: black-box melanoma AUC =",
                  round(auc, 3), "(n =", int(ok.sum()),
                  "; pre-registered gate >= 0.75)", flush=True)
    print("[" + tag + "] wrote", pred_csv, "and weights", flush=True)
    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
print("DONE", flush=True)


In [ ]:
# Paper 2: V2 censored-training test - CELL 2 of 2
# Run AFTER cell 1 (cell 1 writes /kaggle/working/train_v2_censored.py)
import glob
import hashlib
import os
import subprocess
import sys
import time
import zipfile

try:
    LOCAL_TEST
except NameError:
    LOCAL_TEST = False
if LOCAL_TEST:
    WORK = os.environ["P2_OUT"]
    EASY = os.environ["P2_EASY_DIR"]
    SCRIPT = "train_v2_censored.py"
else:
    WORK = "/kaggle/working"
    EASY = "/tmp/easy_images"
    SCRIPT = WORK + "/train_v2_censored.py"
# backbone:seed:pi. pi = 0 and pi = 1 (the primary test) run first.
JOBS_PER_GPU = [
    "effb4:0:0,effb4:0:1,effb4:2:0,effb4:0:0.5,effb4:2:0.5",   # GPU 0
    "effb4:1:0,effb4:1:1,effb4:2:1,effb4:1:0.5",               # GPU 1
]
if LOCAL_TEST:
    JOBS_PER_GPU = ["effb4:0:0,effb4:0:1", "effb4:0:0.5"]

# 1) GPUs
if LOCAL_TEST:
    gpus = ["GPU 0 (test)", "GPU 1 (test)"]
else:
    out = subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                         text=True).stdout
    gpus = [line for line in out.splitlines() if line.strip()]
print("\n".join(gpus) or "NO GPU FOUND")
assert gpus, "Turn on GPU: Settings -> Accelerator -> GPU T4 x2"
assert os.path.exists(SCRIPT), "Run cell 1 first"

# 1b) Check that cell 1 arrived undamaged. This catches copy/paste
#     damage in seconds, before any download or GPU time is spent.
EXPECTED_FP = "129fc9b577f7"
chk = subprocess.run([sys.executable, "-m", "py_compile", SCRIPT],
                     capture_output=True, text=True)
with open(SCRIPT) as fh:
    code_text = fh.read()
squeezed = "".join(code_text.split()).encode()
fp = hashlib.md5(squeezed).hexdigest()[:12]
print("cell 1 check:", len(code_text.splitlines()), "lines | compiles:",
      chk.returncode == 0, "| fingerprint:", fp, "| expected:",
      EXPECTED_FP)
if chk.returncode != 0 or fp != EXPECTED_FP:
    print(chk.stderr[-1500:])
    raise RuntimeError("Cell 1 is damaged or is not the tested version."
                       " Import paper2_v2_censored.ipynb instead of"
                       " pasting.")
print("cell 1 check: OK")


# 2) Download the 248 EASY images (ISIC collection 166)
def n_images():
    found = glob.glob(EASY + "/**/ISIC_*.jp*g", recursive=True)
    return len({os.path.basename(p) for p in found})


if not LOCAL_TEST and n_images() != 248:
    os.makedirs(EASY, exist_ok=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "isic-cli"], check=False)
    r = subprocess.run(["isic", "image", "download", "-l", "0",
                        "-c", "166", EASY],
                       capture_output=True, text=True, timeout=3600)
    print("isic-cli:", (r.stdout + r.stderr)[-500:])
    if n_images() != 248:
        print("isic-cli gave", n_images(), "images -> using the API")
        import requests
        import pandas as pd
        url = ("https://api.isic-archive.com/api/v2/images/search/"
               "?collections=166&limit=100")
        rows = []
        while url:
            page = requests.get(url, timeout=60).json()
            for res in page["results"]:
                iid = res["isic_id"]
                path = EASY + "/" + iid + ".jpg"
                if not os.path.exists(path):
                    img_url = res["files"]["full"]["url"]
                    data = requests.get(img_url, timeout=120).content
                    with open(path, "wb") as fh:
                        fh.write(data)
                clin = res.get("metadata", {}).get("clinical", {})
                rows.append({"isic_id": iid,
                             "diagnosis_1": clin.get("diagnosis_1")})
            url = page.get("next")
        pd.DataFrame(rows).to_csv(EASY + "/metadata_api.csv",
                                  index=False)
print("EASY images available:", n_images())
assert LOCAL_TEST or n_images() == 248, "EASY download incomplete"

# 3) Train + predict: one job list per GPU, in parallel
if len(gpus) >= 2:
    lists = JOBS_PER_GPU
else:
    lists = [",".join(JOBS_PER_GPU)]
jobs = []
for k, joblist in enumerate(lists):
    env = dict(os.environ)
    env["P2_JOBS"] = joblist
    env["P2_EASY_DIR"] = EASY
    env["P2_OUT"] = WORK
    if len(gpus) >= 2 and not LOCAL_TEST:
        env["CUDA_VISIBLE_DEVICES"] = str(k)
    logf = WORK + "/stdout_v2_gpu" + str(k) + ".txt"
    proc = subprocess.Popen([sys.executable, "-u", SCRIPT],
                            stdout=open(logf, "w"),
                            stderr=subprocess.STDOUT, env=env)
    jobs.append((k, proc, logf))
print("started", len(jobs), "job list(s) on", min(len(gpus), 2),
      "GPU(s)", flush=True)
KEYS = ("==", "'epoch': 40", "EASY", "wrote", "DONE", "Error",
        "Traceback", "skip")
seen = {k: 0 for k, _, _ in jobs}
while True:
    for k, proc, logf in jobs:
        lines = open(logf).read().splitlines()
        for line in lines[seen[k]:]:
            if any(s in line for s in KEYS):
                print("[gpu" + str(k) + "]", line, flush=True)
        seen[k] = len(lines)
    if all(proc.poll() is not None for _, proc, _ in jobs):
        break
    time.sleep(5 if LOCAL_TEST else 60)
for k, proc, _ in jobs:
    print("gpu" + str(k), "exit code:", proc.returncode,
          "(0 = success)")

# 4) Zip the small outputs (weights stay in the notebook output)
files = sorted(glob.glob(WORK + "/easy_preds_v2_*.csv")
               + glob.glob(WORK + "/log_full_v2_*.csv")
               + glob.glob(WORK + "/train_labels_v2_*.csv"))
with zipfile.ZipFile(WORK + "/p2_v2.zip", "w",
                     zipfile.ZIP_DEFLATED) as z:
    for f in files:
        z.write(f, os.path.basename(f))
n_exp = 3 * sum(len(j.split(",")) for j in JOBS_PER_GPU)
print("\np2_v2.zip:", len(files), "files (expect", n_exp, ")")
weights = glob.glob(WORK + "/weights/*.pt")
print("weights:", sorted(os.path.basename(p) for p in weights))
